In [5]:
import torch
import torch.nn as nn

def sinusoidal_pe(max_len, d_model):
    pe = torch.zeros(max_len, d_model)                         # (max_len, d_model)
    pos = torch.arange(max_len).unsqueeze(1).float()           # (max_len, 1)
    two_k = torch.arange(0, d_model, 2).float()                # (d_model/2,)  = 2k
    div = torch.pow(10000.0, two_k / d_model)                  # (d_model/2,)  = 10000^(2k/d)
    pe[:, 0::2] = torch.sin(pos / div)                         # çift sütunlar: sin
    pe[:, 1::2] = torch.cos(pos / div)                         # tek sütunlar: cos
    return pe

pe = sinusoidal_pe(60, 32)                                     # dosyadaki ısı haritası ayarı: d=32, 60 pozisyon
print("PE shape:", pe.shape)
print("p_1 (d=4 örneği):", sinusoidal_pe(2, 4)[1])

# Öğrenilmiş konum gömlemesi
learned = nn.Embedding(60, 32)                                 # ağırlık: (60, 32)
positions = torch.arange(6)                                    # (6,)
print("learned:", learned(positions).shape)

# Girdi = token gömlemesi + konum
tok_emb = torch.randn(6, 32)                                   # 6 token
h = tok_emb + pe[:6]                                           # (6,32)+(6,32) -> (6,32)
print("h shape:", h.shape)

PE shape: torch.Size([60, 32])
p_1 (d=4 örneği): tensor([0.8415, 0.5403, 0.0100, 0.9999])
learned: torch.Size([6, 32])
h shape: torch.Size([6, 32])


In [6]:
sent1 = ["I", "walk", "my", "dog", "every", "day"]
sent2 = ["every", "day", "I", "walk", "my", "dog"]

def offsets(sent, focus):
    f = sent.index(focus)                       # odak pozisyonu (0 tabanlı)
    return {w: j - f for j, w in enumerate(sent) if j != f}

print(offsets(sent1, "dog"))
print(offsets(sent2, "dog"))

{'I': -3, 'walk': -2, 'my': -1, 'every': 1, 'day': 2}
{'every': -5, 'day': -4, 'I': -3, 'walk': -2, 'my': -1}


In [3]:
import torch, math

def rotate(v, pos, theta=0.1):
    a = pos * theta                                         # pozisyona bağlı açı
    R = torch.tensor([[math.cos(a), -math.sin(a)],
                      [math.sin(a),  math.cos(a)]])         # (2,2) dönme matrisi
    return R @ v                                            # (2,)

q = torch.tensor([1.0, 0.5])
k = torch.tensor([0.3, 0.8])

# Aynı fark (2), farklı mutlak pozisyonlar:
s1 = rotate(q, 5)  @ rotate(k, 3)
s2 = rotate(q, 15) @ rotate(k, 13)
print(round(s1.item(), 5), round(s2.item(), 5))

0.81518 0.81518


In [4]:
# Yorum: Pozisyonlar (5,3) ve (15,13) farklı, ama fark 2 olduğundan skor aynı. Skor yalnızca göreli mesafeye bağlı kalıyor.